# 01 — Hillstrom: EDA, randomization check & ATE (Day 1)

Verify the e-mail experiment is randomized and measure the average effect,
**before** any uplift modeling. Everything here is a baseline the models must beat.

- Dataset: MineThatData Hillstrom e-mail experiment — 64,000 customers, 3 arms, 14-day window.
- Structured spec: [`docs/datasets/hillstrom.md`](../../docs/datasets/hillstrom.md)
- Run cells top to bottom. Cells marked **TODO [YOU]** need your judgement.

In [ ]:
# Setup: make the repo root importable so `packages.*` resolves from the notebook.
import pathlib
import sys

ROOT = pathlib.Path.cwd().resolve()
for _ in range(8):
    if (ROOT / "pyproject.toml").exists():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

import pandas as pd

from packages.causal.diagnostics import ate_table, smd_table
from packages.datasets.hillstrom import (
    FEATURES,
    LEAKY_COLUMNS,
    OUTCOMES,
    arm_summary,
    fetch_hillstrom,
    load_hillstrom_raw,
)

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 50)
print("features:", FEATURES)
print("outcomes:", OUTCOMES)
print("must never leak:", LEAKY_COLUMNS)

## 1. Raw data

Untouched reference: 64,000 rows, 12 columns, all three arms.

In [ ]:
raw = load_hillstrom_raw()
print("shape:", raw.shape)
print("dtypes:")
display(raw.dtypes)
raw.head()

## 2. Arm sizes and outcome rates

**TODO [YOU]** — read the table below and note:

- Are the three arms ~1/3 each? (expect 21,306 / 21,307 / 21,387)
- Which arm has the higher `visit` and `conversion` rate?
- Why is `mean_spend` so small? (zero-inflation: only 578 of 64,000 customers spend > $0)

Write your answers here:

In [ ]:
arm_summary(raw)

## 3. Choose the binary experiment

**TODO [YOU]** — pick the comparison arm from the lift you just saw, and write down why.

Mens beats Womens on every outcome in the raw rates, so the default is `mens`
(Mens E-Mail vs No E-Mail). Change `ARM` only if your reasoning says otherwise.

In [ ]:
ARM = "mens"          # "mens" or "womens"
df = fetch_hillstrom(arm=ARM)
print(df["segment"].value_counts())
print("treated share:", round(df["treatment"].mean(), 4))
print("rows:", len(df), " conversions:", int(df["conversion"].sum()))
df.head()

## 4. Randomization check — standardized mean differences

Spec: flag any `|SMD| > 0.1`. Each categorical level is compared one-vs-rest.

**TODO [YOU]** — is the sample balanced? If anything is flagged, would you worry,
and does a flag this size matter at n ≈ 42k?

In [ ]:
smd = smd_table(df, FEATURES)
print("flagged (|SMD| > 0.1):", int(smd["flagged"].sum()), "of", len(smd))
smd

## 5. Average treatment effect (with 95% CI)

Treated minus control, for each outcome. This is the number every uplift model
must beat — note that random targeting scores ~0 here.

**TODO [YOU]** — for each outcome write the ATE, the CI, and whether it is
statistically distinguishable from 0. Then sanity-check: does `visit` move more
than `conversion`, as expected?

In [ ]:
ate = ate_table(df, OUTCOMES)
ate

## 6. Profit assumptions — TODO [YOU]

Fill these in and copy them into `configs/business_assumptions.yaml` and the README.
Every one of these must be *stated*, not hidden.

| Assumption | Value | Why |
| --- | --- | --- |
| Reporting currency | **USD** (fixed) | Hillstrom spend is already USD |
| Gross margin on incremental spend | ________ % | converts spend to profit |
| Cost per contact (e-mail) | ________ USD | fixed send cost per customer |
| Value per conversion (Criteo only) | ________ USD | Criteo records no spend |

Note the unit trap: the profit formula is
`profit(k) = v · Δconv(k) − c · n(k)`, so `v` and `c` must be in the same currency
(both USD here) and be per-conversion / per-contact respectively.

## Next checks (Day 1 → Day 2 handoff)

- [ ] SMD table reviewed, flags explained (or "none, balanced" written down)
- [ ] ATE table reviewed and interpreted in words
- [ ] Binary arm chosen and justified
- [ ] Profit assumptions written into `configs/business_assumptions.yaml` + README
- [ ] Spend distribution looked at (histogram of `spend[spend > 0]`) — optional, useful
- [ ] Then Day 2: response baseline vs S-/T-/X-learners and the first Qini curves

Optional, when you want to persist a clean copy for Day 2:

```python
# from packages.datasets.hillstrom import PROCESSED_DIR
# PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
# df.to_parquet(PROCESSED_DIR / f"hillstrom_{ARM}_binary.parquet", index=False)
```